In [ ]:
import matplotlib.pyplot as plt
import scipp as sc
from trex.instrument import Instrument
import scipp.constants as const
from scippneutron.tof import chopper_cascade

T_OFFSET = sc.scalar(1.7, unit="ms")
central_wavelength = sc.scalar(2.5, unit="Å")
rrm: int = 8  # repetition rate multiplication factor
mode = "High Flux"  # Chopper mode


trex = Instrument(wavelength=central_wavelength, rrm=rrm, mode=mode, t_offset=T_OFFSET)
res = trex.model.run()

# Squeeze the pulse dimension since we only have one pulse
events_at_sample = res["Monitor at Sample"].data.squeeze()
# Remove the events that don't make it to the detector
events_at_sample = events_at_sample[~events_at_sample.masks["blocked_by_others"]]

In [ ]:
fig, ax = plt.subplots()
# tof_sample = events_at_sample.hist(wavelength=200, tof=500).plot(norm='log', ax=ax)
toa_sample = events_at_sample.hist(wavelength=800, toa=1500).plot(
    norm="log", cbar=True, ax=ax, cmap="jet"
)
ax.set_title("TOA at sample position\n" + str(trex), fontsize=8)
# TOF line
line_tof = sc.linspace("tof", 70, 140, 100, unit="ms").to(unit="us")
line_wavelength_tof = (
    const.h / const.m_n / trex.monitors["Monitor at Sample"].distance * line_tof
).to(unit="Å")
ax.plot(line_tof, line_wavelength_tof, label="lambda=h/(m*L_sample)*TOF", alpha=0.2)

# # TOA line
line_toa = sc.linspace("toa", 70, 140, 100, unit="ms").to(unit="us")
line_wavelength_toa = (
    const.h
    / const.m_n
    / trex.monitors["Monitor at Sample"].distance
    * (line_toa - T_OFFSET.to(unit="us"))
).to(unit="Å")
ax.plot(
    line_toa, line_wavelength_toa, label="lambda=h/(m*L_sample)*(TOA-1.7 ms)", alpha=0.2
)

line_wavelength_toa = (
    const.h
    / const.m_n
    / trex.monitors["Monitor at Sample"].distance
    * (line_toa - sc.scalar(3.5, unit="ms").to(unit="us"))
).to(unit="Å")
ax.plot(
    line_toa, line_wavelength_toa, label="lambda=h/(m*L_sample)*(TOA-3.5 ms)", alpha=0.2
)

ax.legend()
ax.grid(alpha=0.6)
fig.tight_layout()

In [ ]:
frames = chopper_cascade.FrameSequence.from_source_pulse(
    time_min=sc.scalar(0.0, unit="ms"),
    time_max=sc.scalar(4.0, unit="ms"),  # ESS pulse is 3 ms, but it has a tail
    wavelength_min=sc.scalar(0.0, unit="angstrom"),
    wavelength_max=sc.scalar(4.0, unit="angstrom"),
)
frames = frames.chop(trex.chopper_cascade.values())
at_sample = frames.propagate_to(trex.monitors["Monitor at Sample"].distance)

fig, ax = at_sample.draw(transpose=True)
ax.set_title("Frame propagation\n" + str(trex), fontsize=9)
ax.legend()
fig.tight_layout()

In [ ]:
fig, ax = frames.acceptance_diagram()
ax.set_title("Chopper acceptance diagram\n" + str(trex), fontsize=9)
ax.legend()
fig.tight_layout()

In [ ]:
from scippneutron.tof.chopper_cascade import FrameSequence
from dataclasses import dataclass


@dataclass
class SubframeVertex:
    distance: sc.Variable
    time: sc.Variable
    wavelength: sc.Variable


def acceptance_paths(
    *,
    frames: FrameSequence,
    time_unit: str = 'ms',
    wavelength_unit: str = 'angstrom',
) -> dict[str, dict]:
    frame_paths = {}
    for i, frame in enumerate(frames):
        subframe_paths = []
        for isub, subframe in enumerate(frame.subframes):
            vertex = SubframeVertex(
                distance=frame.distance,
                time = subframe.time.to(unit=time_unit, copy=False),
                wavelength = subframe.wavelength.to(unit=wavelength_unit, copy=False)
            )
            subframe_paths.append(vertex)
        frame_paths[f"{frame.distance:c}"] = subframe_paths
    return frame_paths

In [ ]:
display(acceptance_paths(frames=frames).keys())
subframe_vertexes = acceptance_paths(frames=frames)
subframe_vertexes['162.005 m']

In [ ]:
import tof

source = tof.Source(facility='ess', neutrons=50_000_000)
source

In [ ]:
birth_time_limit = sc.scalar(40., unit='µs')
wavelength_limit = sc.scalar(4., unit='angstrom')

binned = source.data.bin(birth_time=100, wavelength=100)['birth_time', :birth_time_limit]['wavelength', :wavelength_limit]
binned = binned.bin(birth_time=100, wavelength=200)
binned

In [ ]:
import numpy as np

def get_points(da: sc.DataArray, *, xcoord_name: str = 'wavelength', ycoord_name: str = 'birth_time') -> np.ndarray:
    x_coord = da.coords[xcoord_name] if not da.coords.is_edges(xcoord_name) else sc.midpoints(da.coords[xcoord_name])
    y_coord = da.coords[ycoord_name] if not da.coords.is_edges(ycoord_name) else sc.midpoints(da.coords[ycoord_name])
    sizes = {**x_coord.sizes, **y_coord.sizes}
    x_coord = x_coord.broadcast(sizes=sizes).copy(deep=True)
    y_coord = y_coord.broadcast(sizes=sizes).copy(deep=True)
    # Overwriting unit to make a stack using scipp operator...
    x_coord.unit = 'dimensionless'
    y_coord.unit = 'dimensionless'
    xy_stack = sc.concat([x_coord, y_coord], dim='i').flatten(dims=sizes.keys(), to='pos').transpose()
    return xy_stack.values

In [ ]:
example_vertex = subframe_vertexes['31.964 m'][0]
example_vertex

In [ ]:
from matplotlib.path import Path 
import numpy as np

def mask_paths(*, da: sc.DataArray, vertex: SubframeVertex) -> sc.DataArray:
    wav_time_points = get_points(da)
    vx = vertex.wavelength.values
    vy = vertex.time.values
    verts = np.column_stack([vx, vy])
    path = Path(verts)
    sizes = da.sizes
    dims = sizes.keys()
    inside = sc.array(
        dims=dims,
        values=path.contains_points(wav_time_points).reshape(tuple(sizes.values())),
    )
    return da.assign_masks({f"{vertex.distance:c}": ~inside})


masked = mask_paths(da=binned, vertex=example_vertex)
masked

In [ ]:
mask_distance = next(iter(masked.masks.keys()))
display((~next(iter(masked.masks.values()))).sum())  # Number of valid bins
plot = (binned.hist().plot(norm='log', title='Original Source') + masked.hist().plot(norm='log', title=f'Masked Source at {mask_distance}'))
plot

In [ ]:
# from matplotlib.path import Path
# import numpy as np

# frame=frames[-1].propagate_to(trex.source.distance)


# for subframe in frame.subframes:
#     x = subframe.time
#     y = subframe.wavelength
#     x_unit = x.unit
#     y_unit = y.unit
#     # x_max = max(x_max, x.max().value)
#     # y_max = max(y_max, y.max().value)

#     poly_path = Path(
#         np.stack((x.values, y.values), axis=1)
#     )